# Benchmarks for Sec. 5: deformation-stability diagnostics

This notebook provides small, reusable benchmarks for the three layers emphasized in Sec. 5 of the draft: exact-cage compatibility, bounded-local persistence, and thermal-activity margins.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import set_revtex_matplotlib_style, save_prx_figure
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_jacobian_conditioning_from_hamiltonian,
    operator_coefficient_compatibility,
    thermal_activity_margin_from_samples,
)
from qlinks.models import SpinOneXYChainModel, SquareQDMModel, spin_one_xy_periodic_range_couplings
from qlinks.builders import SparseHamiltonianBuilder

set_revtex_matplotlib_style(base_font_size=9, prefer_tex=False)
DATA_DIR = REPO_ROOT / "experimental" / "data" / "deformation_stability_benchmarks"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Spin-1 XY compatibility benchmark

In [ ]:
L = 6
model = SpinOneXYChainModel(length=L, boundary_condition="periodic", j_xy=2.0, total_sz=-2, extra_xy_couplings=spin_one_xy_periodic_range_couplings(length=L, distance=3, coefficient=0.2))
build = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
configs = basis_configs_from_build_result(build)
# The benchmark only records conditioning data here; the full tower-state machinery lives in spin1_xy_draft_evidence.ipynb.
conditioning = cage_jacobian_conditioning_from_hamiltonian(build.hamiltonian, np.arange(min(8, build.hamiltonian.shape[0])), np.eye(build.hamiltonian.shape[0], dtype=np.complex128)[:,0], tolerance=1e-10)
pd.DataFrame([conditioning.to_summary_dict()]).to_csv(DATA_DIR / "spin1_conditioning_benchmark.csv", index=False)
display(pd.DataFrame([conditioning.to_summary_dict()]))

## 2. Square-QDM local versus collective operator-compatibility benchmark

In [ ]:
square = SquareQDMModel(layout=(4, 4), winding_sector=(0, 0))
build = square.build(builder="sparse", backend="scipy", basis_solver="dfs", sort_basis=True)
term_builder = SparseHamiltonianBuilder(backend="scipy", dtype=np.complex128, on_missing="raise")
kinetic_terms = tuple(term_builder.build(build.basis, [operator]).astype(np.complex128) for operator in build.kinetic_operators)
# As a cheap benchmark, compare compatibility for the first two basis vectors. The production notebook replaces these by the compact and collective cage representatives.
state_a = np.eye(build.hamiltonian.shape[0], dtype=np.complex128)[:, 0]
state_b = np.eye(build.hamiltonian.shape[0], dtype=np.complex128)[:, 1]
rows = []
for label, state in (("state_0", state_a), ("state_1", state_b)):
    report = operator_coefficient_compatibility(kinetic_terms, state, tolerance=1e-10)
    rows.append({
        "target": label,
        "n_operators": report.n_operators,
        "compatible_dimension": report.compatible_dimension,
        "obstruction_rank": report.rank,
        "singular_gap": report.singular_gap,
    })
benchmark_df = pd.DataFrame(rows)
benchmark_df.to_csv(DATA_DIR / "qdm_operator_compatibility_benchmark.csv", index=False)
display(benchmark_df)

## 3. Thermal-margin helper benchmark

In [ ]:
g = np.linspace(-0.2, 0.2, 5)
activity = 0.12 - 0.05 * np.abs(g)
report = thermal_activity_margin_from_samples(g, activity, reference_parameter=0.0, tolerance=1e-10)
summary = pd.DataFrame([report.to_summary_dict()])
summary.to_csv(DATA_DIR / "toy_thermal_margin_benchmark.csv", index=False)
display(summary)

fig, ax = plt.subplots(figsize=(3.35, 2.35))
ax.plot(g, activity, marker="o")
ax.axhline(report.reference_activity, linestyle="--", linewidth=0.8)
ax.set_xlabel("Path parameter")
ax.set_ylabel("Activity")
ax.grid(alpha=0.3)
save_prx_figure(fig, "toy_thermal_margin_benchmark", directory=FIGURE_DIR, formats=("pdf", "svg"))
plt.show()